# AgentCore + Strands - Zero to Hero (one notebook, runs top to bottom)

A single, incremental notebook that builds up from a 3-line agent to all **7 AgentCore pillars**, a test **harness**, and shows **where each thing appears in the AWS console**. Every AWS call is wrapped so a missing permission prints the exact fix instead of crashing.

**The 7 AgentCore pillars:**
1. **Runtime** - serverless host for your agent
2. **Memory** - short-term and long-term memory across sessions
3. **Code Interpreter** - sandboxed code execution (exact math, data work)
4. **Browser** - managed headless browser for live web
5. **Gateway** - turn your existing APIs into MCP tools
6. **Identity** - who can call the agent, and how it authenticates outward
7. **Observability** - traces, metrics, and logs in CloudWatch

**How to run:** top to bottom. The first cell installs everything into *this* kernel with `%pip` (no separate kernel setup needed). Region is `us-east-1`. Models are Claude **Haiku 4.5** (fast/cheap) and **Sonnet 4.5** (stronger).

**The one rule that makes this painless:** a helper named `safe(...)` wraps every AWS create/deploy call. If your IAM principal lacks a permission, you get a one-line `SKIP` with the exact policy to attach, and the notebook keeps going. Nothing hard-crashes.


## Step 0 - Install (into this kernel)

`%pip` is the Jupyter magic that installs into the **currently running** kernel. That is the fix for the classic "I installed it but the import fails" problem. Use `%pip` for installs and `!` only for shell commands.


In [ ]:
# %pip installs into THIS kernel (works the same in Jupyter, VS Code, and Colab).
%pip install -q -U strands-agents "strands-agents[a2a]" strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit boto3 pydantic
print("\nInstalled. If a kernel-restart banner appears in Colab, restart and re-run from here.")

In [ ]:
# Optional extras used only by two later cells (Browser pillar). Safe to run now or skip.
%pip install -q playwright nest_asyncio
# Browser also needs a one-time chromium download (shell command -> use !):
# !playwright install chromium
print("optional browser deps installed (chromium download is commented; run it before the Browser cell)")

## Step 1 - The permissions you need (read this once)

Most "I can't create memory / guardrails via code" errors are an **IAM permission gap on your principal**, not a code bug. Attach these two AWS managed policies to the IAM user or role you run as:

| Policy | Unlocks |
|---|---|
| `AmazonBedrockFullAccess` | Model access (Converse/InvokeModel) **and guardrail creation** (`bedrock:CreateGuardrail`) |
| `BedrockAgentCoreFullAccess` | All AgentCore control-plane: Memory, Gateway, Identity, Runtime, Code Interpreter, Browser |

Deploying to Runtime from code (the `Runtime().launch()` / `agentcore` CLI path) also builds a container, so the **caller** needs extra IAM/CodeBuild/ECR/S3/Logs permissions scoped to `bedrock-agentcore-*` resources. The exact policy JSON is in the Appendix at the bottom (copied from the AWS docs).

The next cell **tests** what you actually have and tells you precisely what to add. You do not have to read a policy and guess.

> Reference: AgentCore IAM permissions - docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html


## Step 2 - Config and the `safe()` wrapper

`safe(label, fn, needs=...)` runs an AWS call. On success it returns the result; on a permission or service error it prints a clear `SKIP` line with the fix and returns `None`, so later cells can check and move on.


In [ ]:
import os, json, time, uuid, logging
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
REGION = os.environ["AWS_DEFAULT_REGION"]

MODEL_FAST   = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # cheap, fast (the 'us.' prefix is required)
MODEL_STRONG = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"  # stronger reasoning

import boto3
from botocore.exceptions import ClientError, BotoCoreError

def safe(label, fn, needs=None):
    """Run an AWS call; never crash. Print a clear fix if a permission is missing."""
    try:
        out = fn()
        print(f"OK   | {label}")
        return out
    except (ClientError, BotoCoreError) as e:
        code = getattr(e, "response", {}).get("Error", {}).get("Code", type(e).__name__)
        print(f"SKIP | {label}  ({code})")
        if needs:
            print(f"       fix: attach {needs} to your IAM principal, then re-run this cell.")
        return None
    except Exception as e:
        print(f"SKIP | {label}  ({type(e).__name__}: {e})")
        return None

print("region:", REGION, "| fast:", MODEL_FAST.split('.')[-1], "| strong:", MODEL_STRONG.split('.')[-1])

## Step 3 - Preflight: credentials, model access, and a permission probe

This checks who you are, that the two models are usable, and which AgentCore capabilities your principal can use. Cheap `List`/`Converse` calls only. Read the column of `OK` / `SKIP` lines and fix anything you plan to use.


In [ ]:
# Who am I?
sts = boto3.client("sts", region_name=REGION)
ident = safe("credentials (sts:GetCallerIdentity)", sts.get_caller_identity)
if ident:
    print("       account:", ident["Account"], "| arn:", ident["Arn"])

# Can I call the models? (smallest possible Converse)
brt = boto3.client("bedrock-runtime", region_name=REGION)
def _ping(model_id):
    r = brt.converse(modelId=model_id,
                     messages=[{"role": "user", "content": [{"text": "Reply with the single word: ok"}]}],
                     inferenceConfig={"maxTokens": 5})
    return r["output"]["message"]["content"][0]["text"]
print(); print("Model access:")
safe(f"Haiku 4.5 invoke", lambda: _ping(MODEL_FAST), needs="AmazonBedrockFullAccess + model access in the Bedrock console")
safe(f"Sonnet 4.5 invoke", lambda: _ping(MODEL_STRONG), needs="AmazonBedrockFullAccess + model access in the Bedrock console")

# Which AgentCore + guardrail permissions do I have? (List calls are safe and cheap)
print(); print("Capability permissions:")
ctrl = boto3.client("bedrock-agentcore-control", region_name=REGION)
bdr  = boto3.client("bedrock", region_name=REGION)
safe("Memory (bedrock-agentcore:ListMemories)",        lambda: ctrl.list_memories(maxResults=1),                 needs="BedrockAgentCoreFullAccess")
safe("Gateway (bedrock-agentcore:ListGateways)",       lambda: ctrl.list_gateways(maxResults=1),                 needs="BedrockAgentCoreFullAccess")
safe("CodeInterpreter (ListCodeInterpreters)",         lambda: ctrl.list_code_interpreters(maxResults=1),        needs="BedrockAgentCoreFullAccess")
safe("Browser (ListBrowsers)",                         lambda: ctrl.list_browsers(maxResults=1),                 needs="BedrockAgentCoreFullAccess")
safe("Identity (ListApiKeyCredentialProviders)",       lambda: ctrl.list_api_key_credential_providers(maxResults=1), needs="BedrockAgentCoreFullAccess")
safe("Runtime (ListAgentRuntimes)",                    lambda: ctrl.list_agent_runtimes(maxResults=1),           needs="BedrockAgentCoreFullAccess")
safe("Guardrails (bedrock:ListGuardrails)",            lambda: bdr.list_guardrails(maxResults=1),                needs="AmazonBedrockFullAccess")
print("\nDone. Any SKIP above tells you exactly what to attach. The notebook still runs end to end.")

# Part A - Strands basics (ease in first)

Before any AWS services, get comfortable with the agent itself. Strands is model-driven: you give the model a system prompt and some tools, and its **agent loop** decides when to call a tool, reads the result, and continues until it has an answer.


## A1 - The smallest possible agent

Three lines: pick a model, create the agent, call it. `str(result)` is the final text.


In [ ]:
from strands import Agent

agent = Agent(model=MODEL_FAST, system_prompt="You are a concise assistant.")
result = agent("In one sentence, what is Amazon Bedrock AgentCore?")
print(str(result))

## A2 - Add a custom tool

A tool is just a Python function with the `@tool` decorator. The **docstring** becomes the tool description and the **type hints** become its input schema. The model decides when to call it.


In [ ]:
from strands import tool

@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())

agent = Agent(model=MODEL_FAST, tools=[word_count],
              system_prompt="Use tools when they help. Be concise.")
result = agent("How many words are in this sentence: the quick brown fox jumps?")
print(str(result))
print("\ntools the agent had:", list(agent.tool_names))

## A3 - Structured output

When you need typed data rather than prose, define a Pydantic model and call `agent.structured_output(Model, prompt)`. You get a validated object back.


In [ ]:
from pydantic import BaseModel, Field

class CityFact(BaseModel):
    city: str = Field(description="city name")
    country: str = Field(description="country name")
    fun_fact: str = Field(description="one short fun fact")

agent = Agent(model=MODEL_FAST)
fact = agent.structured_output(CityFact, "Give me a fact about Bengaluru.")
print(type(fact).__name__, "->", fact.city, "|", fact.country)
print("fun fact:", fact.fun_fact)

## A4 - State

An agent can carry `state`: a dict that travels with it and that hooks and tools can read. We use it later so memory hooks know **which user** they are serving.


In [ ]:
agent = Agent(model=MODEL_FAST, state={"actor_id": "user-amelia", "session_id": "sess-001"})
print("actor_id from state:", agent.state.get("actor_id"))
print("session_id from state:", agent.state.get("session_id"))

## A5 - Conversation management (keep context bounded)

Long chats overflow the context window and cost more. A conversation manager keeps the window in check. `SlidingWindowConversationManager` keeps the last N messages.


In [ ]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

agent = Agent(model=MODEL_FAST,
              conversation_manager=SlidingWindowConversationManager(window_size=20))
agent("My name is Amelia.")
print(str(agent("What is my name?")))   # still in the window, so it remembers within this chat

## A6 - Hooks (run your code at lifecycle moments)

Hooks let you observe or extend the agent without touching its core logic. Here is a tiny hook that logs every message. The same mechanism powers AgentCore Memory later.


In [ ]:
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent

class LoggingHook(HookProvider):
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(MessageAddedEvent, self.on_message)
    def on_message(self, event: MessageAddedEvent) -> None:
        role = event.message.get("role", "?")
        print(f"   [hook] message added with role={role}")

agent = Agent(model=MODEL_FAST, hooks=[LoggingHook()], system_prompt="Be brief.")
print(str(agent("Say hello in five words.")))

## A7 - Session management (persist a chat to disk, zero AWS)

`FileSessionManager` saves the conversation to local files, so a new agent with the same `session_id` resumes it. This needs no AWS permissions. In production you swap in `S3SessionManager` for concurrency.


In [ ]:
from strands.session import FileSessionManager

sid = "demo-session-aaa"
a1 = Agent(model=MODEL_FAST, session_manager=FileSessionManager(session_id=sid, storage_dir="./_sessions"))
a1("Remember the project code is BLUEJAY.")

# A brand-new agent object, same session id -> it loads the saved history from disk.
a2 = Agent(model=MODEL_FAST, session_manager=FileSessionManager(session_id=sid, storage_dir="./_sessions"))
print(str(a2("What is the project code?")))

## A8 - Snapshots (capture and restore agent state)

`take_snapshot()` captures the agent's messages and state; `load_snapshot()` restores them into another agent. Useful for checkpointing or moving a conversation between processes.


In [ ]:
src = Agent(model=MODEL_FAST)
src("My favourite colour is teal.")
snap = src.take_snapshot(preset="session")    # a Snapshot object (messages + state)
print("snapshot captured:", type(snap).__name__)

restored = Agent(model=MODEL_FAST)
restored.load_snapshot(snap)                  # restore into a fresh agent
print(str(restored("What is my favourite colour?")))

## A9 - Interrupts (human in the loop) - pattern

Interrupts let a tool pause the run to ask a human (for example, to approve an action), then resume with the human's answer. Because it waits for human input, it does not auto-run top to bottom, so this is shown as a pattern, not an executed cell.

```python
from strands import Agent, tool
from strands.types.interrupt import InterruptException

@tool
def refund(order_id: str, amount: float) -> str:
    """Issue a refund. Requires human approval above 100."""
    if amount > 100:
        # Pause and ask a human. The SDK raises an interrupt the caller handles.
        raise InterruptException(name="approve_refund",
                                 reason={"order_id": order_id, "amount": amount})
    return f"Refunded {amount} for {order_id}"

agent = Agent(model=MODEL_FAST, tools=[refund])
try:
    result = agent("Refund 250 for order A-99")
except InterruptException as it:
    decision = input(f"Approve {it.reason}? (yes/no) ")   # a human decides
    # Resume by calling the agent again with the human's response content.
    # See: strandsagents.com/docs/user-guide/concepts/interrupts/
```
> Full resume mechanics: strandsagents.com/docs/user-guide/concepts/interrupts/


# Part B - The 7 AgentCore pillars

Now the AgentCore services. Each pillar below follows the same rhythm: a short why, the code (wrapped in `safe()`), and where it shows up in the AWS console. Creating cloud resources (Memory, Guardrail, credential providers) runs by default and is wrapped, so a missing permission prints the fix and the notebook continues. The one slow step (deploying a container to Runtime) is gated behind a `DEPLOY` flag so a normal run stays fast.


## Pillar 1 - Runtime (serverless host for your agent)

You wrap the agent in `BedrockAgentCoreApp` and mark one function with `@app.entrypoint`. Locally, `app.run()` starts an HTTP server on `http://localhost:8080` (POST `/invocations`, GET `/ping`). To deploy, the toolkit builds a container and runs it on AgentCore Runtime with per-session microVM isolation.


In [ ]:
%%writefile app.py
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent
from strands_tools import calculator

app = BedrockAgentCoreApp()
_agent = None

@app.entrypoint
def invoke(payload, context=None):
    """Entry point that AgentCore Runtime calls. payload is the request JSON."""
    global _agent
    if _agent is None:
        _agent = Agent(
            model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
            tools=[calculator],
            system_prompt="You are a concise assistant.",
        )
    prompt = payload.get("prompt", "Hello")
    result = _agent(prompt)
    return {"result": result.message}

if __name__ == "__main__":
    app.run()  # local: http://localhost:8080  (POST /invocations, GET /ping)

In [ ]:
# Deploying builds and pushes a container (minutes) and needs the AgentCore CLI IAM (see Appendix).
# Keep it opt-in so a normal run stays fast.
DEPLOY = False
launched = None
if DEPLOY:
    from bedrock_agentcore_starter_toolkit import Runtime
    rt = Runtime()
    safe("configure runtime",
         lambda: rt.configure(entrypoint="app.py", agent_name="z2hagent",
                              requirements=["strands-agents", "strands-agents-tools", "bedrock-agentcore"],
                              auto_create_execution_role=True, auto_create_ecr=True, region=REGION),
         needs="BedrockAgentCoreFullAccess + the AgentCore CLI IAM policy (Appendix)")
    launched = safe("launch runtime (container build, minutes)", lambda: rt.launch(),
                    needs="the AgentCore CLI IAM policy (Appendix)")
    if launched:
        print("agent_arn:", getattr(launched, "agent_arn", launched))
        print("invoke:", safe("invoke deployed runtime", lambda: rt.invoke({"prompt": "Say hello in five words."})))
else:
    print("DEPLOY is False (fast path).")
    print("Local test: in a terminal run  python app.py  then POST to http://localhost:8080/invocations")
    print("Set DEPLOY=True (with the Appendix IAM) to deploy from here, or use the CLI below.")

**CLI alternative** (same result as the toolkit above, run in a terminal in your venv):

```bash
agentcore configure -n z2hagent -e app.py
agentcore launch --local                       # build + run locally in a container
agentcore invoke --local '{"prompt": "hello"}'
agentcore launch                               # deploy to AgentCore Runtime
agentcore status
agentcore invoke '{"prompt": "hello"}'
```

**On the AWS console:** Amazon Bedrock AgentCore (region us-east-1) > **Agent Runtime**. Your deployed agent, its endpoint, and versions appear there. Logs land in CloudWatch under `/aws/bedrock-agentcore/runtimes/*`.


## Pillar 2 - Memory (short-term and long-term)

AgentCore Memory stores conversation events and extracts durable insights. Three built-in strategies:
- **userPreferenceMemoryStrategy** - recurring choices ("prefers window seats")
- **semanticMemoryStrategy** - facts ("the API endpoint is api.example.com")
- **summaryMemoryStrategy** - session summaries

Short-term = the raw turns (`create_event` / `get_last_k_turns`, available immediately). Long-term = the extracted, semantically searchable memories (`retrieve_memories`, extracted asynchronously). Creating memory needs `BedrockAgentCoreFullAccess`. If your earlier preflight showed a SKIP here, that is the exact gap.


In [ ]:
from bedrock_agentcore.memory import MemoryClient

mc = MemoryClient(region_name=REGION)
STRATEGIES = [
    {"userPreferenceMemoryStrategy": {"name": "UserPreferences",
                                      "namespaces": ["support/{actorId}/preferences"]}},
    {"semanticMemoryStrategy":       {"name": "SemanticFacts",
                                      "namespaces": ["support/{actorId}/facts"]}},
]
memory = safe("create AgentCore Memory (waits for ACTIVE, ~1-2 min)",
              lambda: mc.create_memory_and_wait(name="z2h_memory", strategies=STRATEGIES,
                                                description="Zero to hero demo", event_expiry_days=30),
              needs="BedrockAgentCoreFullAccess")
MEMORY_ID = (memory.get("memoryId") or memory.get("id")) if memory else None
print("MEMORY_ID:", MEMORY_ID)

In [ ]:
# Short-term memory: write a turn, read it back (immediate)
ACTOR, SESSION = "user-amelia", "sess-001"
if MEMORY_ID:
    safe("store event (STM)",
         lambda: mc.create_event(memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION,
                                 messages=[("I prefer window seats and vegetarian meals.", "USER")]))
    turns = safe("read last turns (STM)",
                 lambda: mc.get_last_k_turns(memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION, k=5))
    print("recent turns:", turns)
else:
    print("No MEMORY_ID (permission gap). Skipping STM demo.")

In [ ]:
# Long-term memory: semantic retrieve. Extraction is async, so right after writing it may be empty.
if MEMORY_ID:
    hits = safe("semantic retrieve (LTM)",
                lambda: mc.retrieve_memories(memory_id=MEMORY_ID,
                                             namespace=f"support/{ACTOR}/preferences",
                                             query="seating preference", top_k=3))
    print("LTM hits (may be empty until extraction completes):", hits)

### Wiring Memory into an agent with hooks

The clean pattern (from the AWS Strands + AgentCore guide) keeps memory out of the agent logic and inside hooks. The agent carries `state={"actor_id", "session_id"}`; the hooks read it. Note the case handling: AgentCore event roles are uppercase (USER/ASSISTANT), while Strands messages are lowercase, so we normalize.


In [ ]:
import json
from strands.hooks import (HookProvider, HookRegistry,
                           AgentInitializedEvent, MessageAddedEvent, BeforeInvocationEvent)

class ShortMemoryHook(HookProvider):
    def __init__(self, mc, memory_id):
        self.mc, self.memory_id = mc, memory_id
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AgentInitializedEvent, self.on_init)
        registry.add_callback(MessageAddedEvent, self.on_msg)
    def on_init(self, event):
        turns = self.mc.get_last_k_turns(memory_id=self.memory_id,
                                         actor_id=event.agent.state.get("actor_id"),
                                         session_id=event.agent.state.get("session_id"), k=20)
        if turns:
            lines = [f"{m['role']}: {m['content']}" for t in reversed(turns) for m in t]
            event.agent.system_prompt += "\n\nRecent conversation:\n" + "\n".join(lines)
    def on_msg(self, event):
        last = event.agent.messages[-1]
        role = str(last.get("role", "user")).upper()   # AgentCore expects USER/ASSISTANT
        self.mc.create_event(memory_id=self.memory_id,
                             actor_id=event.agent.state.get("actor_id"),
                             session_id=event.agent.state.get("session_id"),
                             messages=[(json.dumps(last["content"]), role)])

class LongTermMemoryHook(HookProvider):
    def __init__(self, mc, memory_id):
        self.mc, self.memory_id = mc, memory_id
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.on_before)
    def on_before(self, event):
        last = event.agent.messages[-1]
        if str(last.get("role", "")).lower() != "user":   # only act on user turns
            return
        q = last.get("content", "")
        q = json.dumps(q) if isinstance(q, list) else str(q)
        hits = self.mc.retrieve_memories(memory_id=self.memory_id,
                                         namespace=f"support/{event.agent.state.get('actor_id')}/facts",
                                         query=q, top_k=3)
        if hits:
            event.agent.system_prompt += "\n\nRelevant long-term memory:\n" + json.dumps(hits)[:1500]

print("hooks defined")

In [ ]:
# A memory-aware agent. New conversations for the same actor recall prior context.
if MEMORY_ID:
    mem_agent = Agent(model=MODEL_FAST,
                      hooks=[ShortMemoryHook(mc, MEMORY_ID), LongTermMemoryHook(mc, MEMORY_ID)],
                      tools=[word_count],
                      state={"actor_id": ACTOR, "session_id": SESSION},
                      system_prompt="You are a travel assistant. Use any remembered preferences.")
    print(safe("memory-aware agent run",
               lambda: str(mem_agent("Help me book a flight. What seat and meal should I pick?"))))
else:
    print("No MEMORY_ID. Once BedrockAgentCoreFullAccess is attached and memory is created, re-run.")

**Agentic RAG vs automatic RAG.** The `LongTermMemoryHook` above is automatic (retrieves on every user turn). You can also give the agent a retrieval *tool* so it decides when and what to search:

```python
@tool
def recall(query: str) -> list:
    """Search the user's long-term memory for relevant facts or preferences."""
    return mc.retrieve_memories(memory_id=MEMORY_ID,
                                namespace=f"support/{ACTOR}/facts", query=query, top_k=5)
# add recall to the agent's tools=[...]
```

**On the AWS console:** Amazon Bedrock AgentCore > **Memory**. You see the memory resource, its strategies, and (after extraction) stored memories per actor namespace.


## Pillar 3 - Code Interpreter (sandboxed code, exact math)

Give the agent a managed sandbox and it can run real Python for anything that must be exact (money, dates, data). The model writes code; the sandbox runs it; the agent reads the result. Needs `BedrockAgentCoreFullAccess`.


In [ ]:
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

ci = AgentCoreCodeInterpreter(region=REGION)
ci_agent = Agent(model=MODEL_FAST, tools=[ci.code_interpreter],
                 system_prompt="For any calculation, run code in the interpreter and report the exact number.")
print(safe("code interpreter run",
           lambda: str(ci_agent("A plan is 47 seats at 29.99 USD with a 12.5% discount. Compute the exact monthly total.")),
           needs="BedrockAgentCoreFullAccess"))

**On the AWS console:** Code Interpreter sessions are short-lived compute. You see invocation logs in CloudWatch; the tool itself is managed by AgentCore (no standing resource to click unless you create a custom one).


## Pillar 4 - Browser (managed headless browser)

A sandboxed browser lets the agent read live web pages. It needs `playwright` + `nest_asyncio` + a one-time chromium download, plus `BedrockAgentCoreFullAccess`. Live web in a notebook is fragile, so this is opt-in.


In [ ]:
# Prereqs: %pip install playwright nest_asyncio   then   !playwright install chromium
RUN_BROWSER = False
if RUN_BROWSER:
    import nest_asyncio; nest_asyncio.apply()
    from strands_tools.browser import AgentCoreBrowser
    br = AgentCoreBrowser(region=REGION)
    br_agent = Agent(model=MODEL_FAST, tools=[br.browser],
                     system_prompt="Use the browser to look up live information.")
    print(safe("browser run",
               lambda: str(br_agent("Open example.com and tell me its main heading.")),
               needs="BedrockAgentCoreFullAccess + chromium installed"))
else:
    print("RUN_BROWSER is False. Install chromium and set it True to exercise the Browser pillar.")

**On the AWS console:** Amazon Bedrock AgentCore > **Browser tool**. Custom browsers and their sessions appear there; logs in CloudWatch.


## Pillar 5 - Gateway (turn your APIs into MCP tools)

A Gateway exposes existing REST APIs or Lambdas as MCP tools the agent can call, behind a JWT authorizer (Cognito, Entra ID, Okta, and similar). Creating one needs `BedrockAgentCoreFullAccess`, an IAM role the gateway assumes, and your identity provider's details. Because those are environment-specific, this cell only creates a gateway when you fill them in.


In [ ]:
GATEWAY_ROLE_ARN = ""   # an IAM role ARN the gateway assumes (fill in for a real gateway)
IDP_DISCOVERY_URL = ""  # your IdP OpenID config URL
IDP_CLIENT_ID = ""      # an allowed client id
gw = None
if GATEWAY_ROLE_ARN and IDP_DISCOVERY_URL and IDP_CLIENT_ID:
    gw = safe("create gateway",
              lambda: ctrl.create_gateway(name="z2h-gateway", roleArn=GATEWAY_ROLE_ARN,
                  protocolType="MCP", authorizerType="CUSTOM_JWT",
                  authorizerConfiguration={"customJWTAuthorizer": {
                      "discoveryUrl": IDP_DISCOVERY_URL, "allowedClients": [IDP_CLIENT_ID]}}),
              needs="BedrockAgentCoreFullAccess + a gateway IAM role + a JWT IdP")
    if gw:
        print("gatewayId:", gw.get("gatewayId"), "| MCP endpoint:", gw.get("gatewayUrl"))
        # Next: ctrl.create_gateway_target(gatewayIdentifier=gw["gatewayId"], name="...",
        #       targetConfiguration={...}, credentialProviderConfigurations=[...])
        # The target schema depends on your API/Lambda; see the AgentCore Gateway docs.
else:
    print("Fill GATEWAY_ROLE_ARN, IDP_DISCOVERY_URL, IDP_CLIENT_ID to create a Gateway.")

**Connect to a Gateway as MCP tools** (once you have a gateway URL and a bearer token):

```python
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

client = MCPClient(lambda: streamablehttp_client(GATEWAY_URL,
                   headers={"Authorization": f"Bearer {TOKEN}"}))
with client:
    tools = client.list_tools_sync()
    agent = Agent(model=MODEL_FAST, tools=tools)   # your APIs are now agent tools
    print(agent("Use the gateway tools to ..."))
```

**On the AWS console:** Amazon Bedrock AgentCore > **Gateways**. The gateway, its targets, and the MCP endpoint URL appear there.


## Pillar 6 - Identity (who calls the agent, and how it authenticates outward)

Two directions:
- **Inbound** - who is allowed to invoke your agent (IAM, or a JWT from Cognito/Entra/Okta).
- **Outbound** - how your agent authenticates to other services without hardcoding secrets. You register a credential provider once, then a decorator injects the secret at call time.

Creating a provider needs `BedrockAgentCoreFullAccess`.


In [ ]:
cred = safe("create API key credential provider",
            lambda: ctrl.create_api_key_credential_provider(name="z2h-demo-key",
                                                            apiKey="demo-not-a-real-secret"),
            needs="BedrockAgentCoreFullAccess")
if cred:
    print("provider:", cred.get("name") or cred.get("credentialProviderArn"))

**Use the secret without hardcoding it.** A decorator pulls the key from the provider at runtime:

```python
from bedrock_agentcore.identity import requires_api_key

@requires_api_key(provider_name="z2h-demo-key", into="api_key")
async def call_partner_api(api_key: str = ""):
    # api_key is injected by AgentCore Identity. It never appears in your code or repo.
    ...
```

For OAuth downstream services use `@requires_access_token(provider_name=..., scopes=[...], auth_flow="M2M")`.

**On the AWS console:** Amazon Bedrock AgentCore > **Identity**. Credential providers and workload identities appear there.


## Pillar 7 - Observability (traces, metrics, logs)

Observability is **not** fully automatic. Two layers:
1. **In-process metrics** - available right now from `result.metrics` (tokens, latency).
2. **Full traces in CloudWatch** - a one-time setup: enable **CloudWatch Transaction Search**, add the ADOT package (`aws-opentelemetry-distro`), and run the agent under `opentelemetry-instrument`. The Runtime execution role also needs `xray:Put*` and `cloudwatch:PutMetricData` (namespace `bedrock-agentcore`) - both are in the Appendix execution role.


In [ ]:
# Layer 1: per-request metrics, available immediately.
obs_agent = Agent(model=MODEL_FAST, tools=[word_count])
r = obs_agent("How many words: a b c d e?")
m = r.metrics.accumulated_usage
print("tokens -> input:", m["inputTokens"], "| output:", m["outputTokens"], "| total:", m["totalTokens"])

**Layer 2: enable full traces (one time).**

```bash
# 1) Enable CloudWatch Transaction Search (console: CloudWatch > Application Signals > Transaction Search,
#    or via the API). This turns on trace ingestion.
# 2) Add ADOT to your agent's dependencies:
pip install aws-opentelemetry-distro
# 3) Run the agent under the OpenTelemetry instrumentor (AgentCore Runtime wires this when observability is on):
opentelemetry-instrument python app.py
```

**On the AWS console:** **CloudWatch** > **Transaction Search** and **GenAI Observability** show traces and spans for each agent session. Logs are under `/aws/bedrock-agentcore/runtimes/*`.


# Guardrails (safety filters)

Guardrails block unsafe input and output. **Creating one needs `AmazonBedrockFullAccess`** (it grants `bedrock:CreateGuardrail`). If your "create guardrail" calls failed before, a missing `AmazonBedrockFullAccess` is the usual cause - the preflight flags it. You attach a guardrail to the model with `BedrockModel(guardrail_id=...)`.


In [ ]:
gr = safe("create guardrail",
          lambda: bdr.create_guardrail(
              name="z2h-guardrail",
              description="demo guardrail",
              blockedInputMessaging="I can't help with that.",
              blockedOutputsMessaging="I can't help with that.",
              contentPolicyConfig={"filtersConfig": [
                  {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                  {"type": "HATE",     "inputStrength": "HIGH", "outputStrength": "HIGH"}]}),
          needs="AmazonBedrockFullAccess (grants bedrock:CreateGuardrail)")
GUARDRAIL_ID  = gr.get("guardrailId") if gr else None
GUARDRAIL_VER = gr.get("version") if gr else None
print("GUARDRAIL_ID:", GUARDRAIL_ID, "| version:", GUARDRAIL_VER)

In [ ]:
# Attach the guardrail to the model the agent uses.
from strands.models import BedrockModel
if GUARDRAIL_ID:
    guarded = BedrockModel(model_id=MODEL_FAST, region_name=REGION,
                           guardrail_id=GUARDRAIL_ID, guardrail_version=str(GUARDRAIL_VER or "DRAFT"))
    g_agent = Agent(model=guarded, system_prompt="Be helpful and concise.")
    print(safe("guarded agent run", lambda: str(g_agent("Give me one fact about Bengaluru."))))
else:
    print("No guardrail (permission gap). Attach AmazonBedrockFullAccess, then re-run this section.")

**On the AWS console:** Amazon Bedrock > **Guardrails**. You see the guardrail, its versions, and its policies.


# Multi-agent patterns

Three ways to combine agents:
- **Agents as tools** - one coordinator calls specialist agents as if they were tools. Predictable, easy to debug.
- **Graph** - a directed graph with explicit edges. Best when the flow is known and you may want parallel branches.
- **Swarm** - agents hand off to each other autonomously. Flexible, harder to trace.


In [ ]:
# Agents as tools (the robust, common pattern): wrap a specialist agent in a @tool.
@tool
def research(topic: str) -> str:
    """Research a topic and return two or three brief facts."""
    return str(Agent(model=MODEL_FAST, system_prompt="Give 2-3 brief facts.")(topic))

coordinator = Agent(model=MODEL_STRONG, tools=[research],
                    system_prompt="Use the research tool, then write a one-line summary.")
print(safe("agents-as-tools run",
           lambda: str(coordinator("Research Bengaluru and summarize it in one line."))))
# Shortcut: an Agent can also become a tool directly with  specialist.as_tool(name=..., description=...)

In [ ]:
# Graph: explicit two-step flow.
from strands.multiagent import GraphBuilder
b = GraphBuilder()
b.add_node(Agent(model=MODEL_FAST, system_prompt="Restate the question in one line."), "intake")
b.add_node(Agent(model=MODEL_FAST, system_prompt="Answer the question in one line."), "answer")
b.add_edge("intake", "answer")
b.set_entry_point("intake")
b.set_max_node_executions(10)   # this graph is acyclic; the limit just silences a generic warning
graph = b.build()
gres = safe("graph run", lambda: graph("Why is the sky blue?"))
if gres is not None:
    print("status:", gres.status)
    print(str(gres))

In [ ]:
# Swarm: autonomous handoff between specialists.
from strands.multiagent import Swarm
planner  = Agent(model=MODEL_FAST, name="planner",  system_prompt="Break the task into one step, then hand off.")
writer2  = Agent(model=MODEL_FAST, name="writer",   system_prompt="Write the final one-line answer.")
swarm = Swarm([planner, writer2], entry_point=planner, max_handoffs=4, max_iterations=4)
sres = safe("swarm run", lambda: swarm("Give a one-line tip for first-time visitors to Bengaluru."))
if sres is not None:
    print("status:", sres.status)
    print(str(sres))

# A test harness (run cases, check, report)

A harness runs your agent against a set of cases and reports pass/fail plus latency. This one is dependency-free so it always runs. For production evaluation, the **Strands Evals SDK** offers ready-made evaluators (correctness, tool-selection, faithfulness), and **AgentCore Harnesses** run managed evaluations at scale.


In [ ]:
import time

def run_harness(agent, cases):
    rows = []
    for name, prompt, check in cases:
        t0 = time.time()
        out = str(agent(prompt))
        rows.append((name, "PASS" if check(out) else "FAIL", round(time.time() - t0, 2), out[:55]))
    print(f"{'case':<20}{'result':<8}{'sec':<7}preview")
    print("-" * 70)
    for n, res, s, prev in rows:
        print(f"{n:<20}{res:<8}{str(s):<7}{prev}")
    passed = sum(1 for _, res, _, _ in rows if res == "PASS")
    print("-" * 70)
    print(f"{passed}/{len(rows)} passed")
    return rows

harness_agent = Agent(model=MODEL_FAST, tools=[word_count],
                      system_prompt="Be precise and concise.")
cases = [
    ("counts words",  "How many words: the cat sat down?",      lambda o: "5" in o),
    ("knows capital", "Capital of France, one word only.",      lambda o: "paris" in o.lower()),
    ("stays brief",   "Say hi.",                                 lambda o: len(o) < 80),
]
safe("harness run", lambda: run_harness(harness_agent, cases))

# Cleanup (avoid surprise charges)

Delete what this notebook created. Each call is wrapped, and reads its id defensively, so this is safe to run even if some resources were never created.


In [ ]:
_mid = globals().get("MEMORY_ID")
if _mid:
    safe("delete memory", lambda: ctrl.delete_memory(memoryId=_mid))

_gid = globals().get("GUARDRAIL_ID")
if _gid:
    safe("delete guardrail", lambda: bdr.delete_guardrail(guardrailIdentifier=_gid))

safe("delete credential provider", lambda: ctrl.delete_api_key_credential_provider(name="z2h-demo-key"))

_gw = globals().get("gw")
if _gw:
    safe("delete gateway", lambda: ctrl.delete_gateway(gatewayIdentifier=_gw.get("gatewayId")))

# If you deployed a Runtime: rt.destroy()   (toolkit)   or   !agentcore destroy   (CLI)
print("\ncleanup attempted; confirm in the console.")

# Appendix - the exact IAM (copy-paste)

**Caller (you, running this notebook):** attach both managed policies.
- `AmazonBedrockFullAccess` - model access + `bedrock:CreateGuardrail`
- `BedrockAgentCoreFullAccess` - Memory, Gateway, Identity, Runtime, Code Interpreter, Browser

**Deploying to Runtime from code/CLI** also needs this policy on the caller (from the AWS docs; scoped to `bedrock-agentcore-*`):

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {"Sid": "IAMRoleManagement", "Effect": "Allow",
     "Action": ["iam:CreateRole","iam:DeleteRole","iam:GetRole","iam:PutRolePolicy",
                "iam:DeleteRolePolicy","iam:AttachRolePolicy","iam:DetachRolePolicy",
                "iam:TagRole","iam:ListRolePolicies","iam:ListAttachedRolePolicies"],
     "Resource": ["arn:aws:iam::*:role/*BedrockAgentCore*",
                  "arn:aws:iam::*:role/service-role/*BedrockAgentCore*"]},
    {"Sid": "PassRole", "Effect": "Allow", "Action": ["iam:PassRole"],
     "Resource": ["arn:aws:iam::*:role/AmazonBedrockAgentCore*",
                  "arn:aws:iam::*:role/service-role/AmazonBedrockAgentCore*"]},
    {"Sid": "CodeBuild", "Effect": "Allow",
     "Action": ["codebuild:StartBuild","codebuild:BatchGetBuilds","codebuild:ListBuildsForProject",
                "codebuild:CreateProject","codebuild:UpdateProject","codebuild:BatchGetProjects",
                "codebuild:ListProjects"],
     "Resource": "*"},
    {"Sid": "ECR", "Effect": "Allow",
     "Action": ["ecr:CreateRepository","ecr:DescribeRepositories","ecr:GetRepositoryPolicy",
                "ecr:InitiateLayerUpload","ecr:CompleteLayerUpload","ecr:PutImage",
                "ecr:UploadLayerPart","ecr:BatchCheckLayerAvailability","ecr:GetDownloadUrlForLayer",
                "ecr:BatchGetImage","ecr:ListImages","ecr:TagResource","ecr:GetAuthorizationToken"],
     "Resource": "*"},
    {"Sid": "S3", "Effect": "Allow",
     "Action": ["s3:GetObject","s3:PutObject","s3:ListBucket","s3:CreateBucket","s3:PutLifecycleConfiguration"],
     "Resource": ["arn:aws:s3:::bedrock-agentcore-*","arn:aws:s3:::bedrock-agentcore-*/*"]},
    {"Sid": "Logs", "Effect": "Allow",
     "Action": ["logs:GetLogEvents","logs:DescribeLogGroups","logs:DescribeLogStreams"],
     "Resource": ["arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/*",
                  "arn:aws:logs:*:*:log-group:/aws/codebuild/*"]}
  ]
}
```

**Runtime execution role** (the role AgentCore assumes to run the agent; replace account id and region):

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {"Effect": "Allow", "Action": ["logs:CreateLogGroup","logs:CreateLogStream","logs:PutLogEvents",
                                   "logs:DescribeLogStreams","logs:DescribeLogGroups"],
     "Resource": ["arn:aws:logs:us-east-1:123456789012:log-group:/aws/bedrock-agentcore/runtimes/*",
                  "arn:aws:logs:us-east-1:123456789012:log-group:*"]},
    {"Effect": "Allow", "Action": ["xray:PutTraceSegments","xray:PutTelemetryRecords",
                                   "xray:GetSamplingRules","xray:GetSamplingTargets"], "Resource": ["*"]},
    {"Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*",
     "Condition": {"StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}}},
    {"Sid": "WorkloadToken", "Effect": "Allow",
     "Action": ["bedrock-agentcore:GetWorkloadAccessToken",
                "bedrock-agentcore:GetWorkloadAccessTokenForJWT",
                "bedrock-agentcore:GetWorkloadAccessTokenForUserId"],
     "Resource": ["arn:aws:bedrock-agentcore:us-east-1:123456789012:workload-identity-directory/default",
                  "arn:aws:bedrock-agentcore:us-east-1:123456789012:workload-identity-directory/default/workload-identity/agentName-*"]},
    {"Sid": "ModelInvocation", "Effect": "Allow",
     "Action": ["bedrock:InvokeModel","bedrock:InvokeModelWithResponseStream"],
     "Resource": ["arn:aws:bedrock:*::foundation-model/*","arn:aws:bedrock:us-east-1:123456789012:*"]}
  ]
}
```

**Execution role trust policy** (lets AgentCore assume the role):

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Sid": "AssumeRolePolicy", "Effect": "Allow",
    "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
    "Action": "sts:AssumeRole",
    "Condition": {"StringEquals": {"aws:SourceAccount": "123456789012"},
                  "ArnLike": {"aws:SourceArn": "arn:aws:bedrock-agentcore:us-east-1:123456789012:*"}}
  }]
}
```

> Model invocation in the execution role is `bedrock:InvokeModel` (not `bedrock:Converse`). Source: docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html

**Console map (region us-east-1):**
- Runtime, Memory, Gateways, Identity, Browser, Code Interpreter -> Amazon Bedrock AgentCore console
- Guardrails -> Amazon Bedrock console > Guardrails
- Traces and metrics -> CloudWatch > Transaction Search / GenAI Observability
- Logs -> CloudWatch Logs > `/aws/bedrock-agentcore/runtimes/*`
